In [ ]:
import glob
import os
from pathlib import Path

import duckdb as dd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import DATABASE_DIR, HOME_CREDIT_DATA_DIR, HOME_CREDIT_DB
from src.const import (
    APPLICATION_DATE,
    DEFAULT_THRESHOLD,
    DELINQUENCY_THRESHOLD,
    INACTIVITY_PERIOD,
    PROBATION_PERIOD,
    STICKINESS_PERIOD,
    TRANSACTOR_PERIOD,
    WRITE_OFF_THRESHOLD,
)

%load_ext autoreload
%autoreload 2

In [30]:
os.chdir(DATABASE_DIR)
con = dd.connect(HOME_CREDIT_DB)

# **Behavioural Indicator Construction**

A core variable derivation step involves assigning a status code to each account snapshot. To support behavioural segmentation and model feature creation, the following account history indicators are engineered over trailing observation windows:

- Product Type Classifier: Standardised flag to group products (e.g., Classic vs Platinum credit cards) to support segmentation and stratification;
- Months In Arrears/DPD Buckets: Mapping days past due (e.g., 0, 1–30, 31–60, etc.) into categorical delinquency stages;
- Instant Default: Defined according to regulatory default (e.g., >90 DPD or unlikely-to-pay);
- Write-off (WO): Identified via charge-off flags or provisioning rules;
- Probation Flag: Whether the account defaulted in the previous 12 months;
- Cure Flag: Whether the account is no longer in instant default or in probation;
- Default Flag: Whether the account is not in cure;
- Delinquency Indicator: Accounts showing non-performing behaviour but not yet defaulted (e.g., ≥60 DPD);
- Inactivity Flag: True if no customer-initiated activity for 4+ months;
- Transactor Flag: Indicates if the account has shown signs of repayment activity (i.e., not revolving) in the past 2 months;

#### **Loan Term Column**

In [31]:
query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN LOAN_TERM 
"""
con.execute(query)

query = """
   ALTER TABLE credit_card_balance
   ADD COLUMN LOAN_TERM INTEGER;
"""
con.execute(query)

query = f"""
   WITH loan_terms AS (
      SELECT
         SK_ID_CURR,
         SK_ID_PREV,
         MONTHS_BALANCE,
         ROW_NUMBER() OVER (
               PARTITION BY SK_ID_CURR, SK_ID_PREV
               ORDER BY MONTHS_BALANCE
         ) AS loan_term
      FROM credit_card_balance
   )
   
   UPDATE credit_card_balance AS ccb
   SET LOAN_TERM = lt.loan_term
   FROM loan_terms lt
   WHERE ccb.SK_ID_CURR = lt.SK_ID_CURR
   AND ccb.SK_ID_PREV = lt.SK_ID_PREV
   AND ccb.MONTHS_BALANCE = lt.MONTHS_BALANCE;
"""

con.execute(query)

#### **Calendar Date Column**

In [32]:
query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN CALENDAR_DATE 
"""
con.execute(query)

query = f"""
   ALTER TABLE credit_card_balance
   ADD COLUMN CALENDAR_DATE DATE;
"""
con.execute(query)

query = f"""
   UPDATE credit_card_balance
   SET CALENDAR_DATE = DATE '{APPLICATION_DATE}' + (INTERVAL '1 month' * MONTHS_BALANCE);
"""

con.execute(query)

#### **Missed Payment Column**

In [33]:
query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN MISSED_PAYMENT_INDICATOR 
"""
con.execute(query)

query = """
    ALTER TABLE credit_card_balance
    ADD COLUMN MISSED_PAYMENT_INDICATOR INT DEFAULT 0;
"""
con.execute(query)

query = """
    UPDATE credit_card_balance
    SET MISSED_PAYMENT_INDICATOR = (
        CASE 
            WHEN (AMT_TOTAL_RECEIVABLE > 0) AND (AMT_PAYMENT_TOTAL_CURRENT < AMT_INST_MIN_REGULARITY) THEN 1
            WHEN (AMT_TOTAL_RECEIVABLE > 0) AND (AMT_PAYMENT_TOTAL_CURRENT = 0) AND (AMT_INST_MIN_REGULARITY = 0) THEN 1
            ELSE 0
        END
    )
    
"""
con.execute(query)

#### **DPD Column**

In [34]:
query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN DPD 
"""
con.execute(query)

query = """
    ALTER TABLE credit_card_balance
    ADD COLUMN DPD INT DEFAULT 0;
"""
con.execute(query)

query = """
    WITH flagged AS (
        SELECT 
            SK_ID_CURR, SK_ID_PREV, LOAN_TERM,
            MISSED_PAYMENT_INDICATOR,
            SUM(CASE WHEN MISSED_PAYMENT_INDICATOR = 0 THEN 1 ELSE 0 END) OVER (
                PARTITION BY SK_ID_CURR, SK_ID_PREV 
                ORDER BY LOAN_TERM
            ) AS reset_group
        FROM credit_card_balance
    ),
    
    dpd_calc AS (
        SELECT 
            SK_ID_CURR,
            SK_ID_PREV,
            LOAN_TERM,
            MISSED_PAYMENT_INDICATOR,
            ROW_NUMBER() OVER (
                PARTITION BY SK_ID_CURR, SK_ID_PREV, reset_group
                ORDER BY LOAN_TERM
            ) - 1 AS dpd
        FROM flagged
    )

    UPDATE credit_card_balance
    SET DPD = (
        SELECT d.DPD*30
        FROM dpd_calc d
        WHERE credit_card_balance.SK_ID_CURR = d.SK_ID_CURR
        AND credit_card_balance.SK_ID_PREV = d.SK_ID_PREV
        AND credit_card_balance.LOAN_TERM  = d.LOAN_TERM
    );

"""

con.execute(query)

#### **Arrears Column**

In [35]:
query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN MONTHS_IN_ARREARS 
"""
con.execute(query)

query = """
    ALTER TABLE credit_card_balance
    ADD COLUMN MONTHS_IN_ARREARS INT DEFAULT 0;
"""
con.execute(query)

query = """
    UPDATE credit_card_balance
    SET MONTHS_IN_ARREARS = ROUND(DPD / 30, 0)
"""
con.execute(query)

#### **Arrears Staging Column**

In [36]:
query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN ARREARS_STAGE 
"""
con.execute(query)

query = """
    ALTER TABLE credit_card_balance
    ADD COLUMN ARREARS_STAGE VARCHAR(15) DEFAULT 'PERFORMING';
"""
con.execute(query)

query = """
    UPDATE credit_card_balance
    SET ARREARS_STAGE = ( CASE 
        WHEN DPD = 0 THEN '1. PERFORMING'
        WHEN DPD BETWEEN 1 AND 30 THEN '2. 1-30 DPD'
        WHEN DPD BETWEEN 31 AND 90 THEN '3. 31-90 DPD'
        WHEN DPD > 90 THEN '4. 90+ DPD'
    END
    )
"""
con.execute(query)

#### **Instant Default Indicator Column**

In [37]:
query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN INSTANT_DEFAULT_INDICATOR 
"""
con.execute(query)

query = """
    ALTER TABLE credit_card_balance
    ADD COLUMN INSTANT_DEFAULT_INDICATOR BOOLEAN DEFAULT FALSE;
"""
con.execute(query)

query = f"""
    UPDATE credit_card_balance
    SET INSTANT_DEFAULT_INDICATOR = (
        CASE
            WHEN MONTHS_IN_ARREARS >= {DEFAULT_THRESHOLD} THEN TRUE
            ELSE FALSE
        END
    );
"""
con.execute(query)

#### **Write-off Indicator Column**

In [38]:
query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN WO_INDICATOR 
"""
con.execute(query)

query = """
    ALTER TABLE credit_card_balance
    ADD COLUMN WO_INDICATOR BOOLEAN DEFAULT FALSE;
"""
con.execute(query)

query = f"""
    UPDATE credit_card_balance
    SET WO_INDICATOR = (
        CASE
            WHEN MONTHS_IN_ARREARS >= {WRITE_OFF_THRESHOLD} THEN TRUE
            ELSE FALSE
        END
    );
"""
con.execute(query)

#### **Probation Indicator Column**

In [39]:
query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN PROBATION_INDICATOR
"""
con.execute(query)

query = """
    ALTER TABLE credit_card_balance
    ADD COLUMN PROBATION_INDICATOR BOOLEAN DEFAULT FALSE;
"""
con.execute(query)

query = f"""
    UPDATE credit_card_balance t1
    SET PROBATION_INDICATOR = EXISTS (
        SELECT 1 
        FROM credit_card_balance t2
        WHERE t1.SK_ID_CURR = t2.SK_ID_CURR
        AND t1.SK_ID_PREV = t2.SK_ID_PREV
        AND t2.INSTANT_DEFAULT_INDICATOR = TRUE
        AND t2.LOAN_TERM BETWEEN (t1.LOAN_TERM - {PROBATION_PERIOD}) AND (t1.LOAN_TERM - 1)
    );
"""
con.execute(query)

#### **Cure Indicator Column**

In [40]:
query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN CURE_INDICATOR 
"""
con.execute(query)

query = """
    ALTER TABLE credit_card_balance
    ADD COLUMN CURE_INDICATOR BOOLEAN DEFAULT FALSE;
"""
con.execute(query)

query = f"""
    UPDATE credit_card_balance
    SET CURE_INDICATOR = (
        CASE
            WHEN INSTANT_DEFAULT_INDICATOR = FALSE AND PROBATION_INDICATOR = FALSE THEN TRUE
            ELSE FALSE
        END
    );
"""
con.execute(query)

#### **Default Indicator Column**

In [41]:
query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN DEFAULT_INDICATOR 
"""
con.execute(query)

query = """
    ALTER TABLE credit_card_balance
    ADD COLUMN DEFAULT_INDICATOR BOOLEAN DEFAULT FALSE;
"""
con.execute(query)

query = f"""
    UPDATE credit_card_balance
    SET DEFAULT_INDICATOR = NOT CURE_INDICATOR
"""
con.execute(query)

#### **Default Type Column**

In [42]:
query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN DEFAULT_TYPE
"""
con.execute(query)

query = """
    ALTER TABLE credit_card_balance
    ADD COLUMN DEFAULT_TYPE VARCHAR(10) DEFAULT NULL;
"""
con.execute(query)

query = f"""
    UPDATE credit_card_balance
    SET DEFAULT_TYPE = (
        CASE
            WHEN DEFAULT_INDICATOR = TRUE THEN 'D3+'
            ELSE NULL
        END
    );
"""
con.execute(query)

#### **Default Date & Counter Column**

In [43]:
query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN DEFAULT_DATE;
    
    ALTER TABLE credit_card_balance
    ADD COLUMN DEFAULT_DATE DATE DEFAULT NULL;
"""
con.execute(query)

query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN DEFAULT_COUNTER;

    ALTER TABLE credit_card_balance
    ADD COLUMN DEFAULT_COUNTER INTEGER DEFAULT 0;
"""
con.execute(query)

query = f"""
    WITH lead_cure AS (
        SELECT 
            SK_ID_CURR, SK_ID_PREV,  LOAN_TERM, 
            CALENDAR_DATE,
            DEFAULT_INDICATOR,
            LEAD(CURE_INDICATOR,1,NULL) OVER (
                PARTITION BY SK_ID_CURR, SK_ID_PREV
                ORDER BY LOAN_TERM DESC
            ) AS PRIOR_CURE
        FROM credit_card_balance
    ),
    
    default_starts AS (
        SELECT 
            SK_ID_CURR, SK_ID_PREV, LOAN_TERM,
            CALENDAR_DATE AS DEFAULT_START_DATE,
            1 AS NEW_DEFAULT_FLAG
        FROM lead_cure
        WHERE DEFAULT_INDICATOR = TRUE AND PRIOR_CURE = TRUE
    ),
    
    defaults_filled AS (
    SELECT 
        f.SK_ID_CURR,
        f.SK_ID_PREV,
        f.LOAN_TERM,
        f.CALENDAR_DATE,
        SUM(COALESCE(d.NEW_DEFAULT_FLAG, 0)) OVER (
            PARTITION BY f.SK_ID_CURR, f.SK_ID_PREV
            ORDER BY f.LOAN_TERM
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS DEFAULT_COUNTER,
        MAX(d.DEFAULT_START_DATE) OVER (
            PARTITION BY f.SK_ID_CURR, f.SK_ID_PREV
            ORDER BY f.LOAN_TERM
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS FILLED_DEFAULT_DATE
    FROM lead_cure f
    LEFT JOIN default_starts d
        ON f.SK_ID_CURR = d.SK_ID_CURR
        AND f.SK_ID_PREV = d.SK_ID_PREV
        AND f.LOAN_TERM = d.LOAN_TERM
    )

    UPDATE credit_card_balance AS ccb
    SET DEFAULT_DATE = df.FILLED_DEFAULT_DATE,
        DEFAULT_COUNTER = df.DEFAULT_COUNTER
    FROM defaults_filled df
    WHERE ccb.SK_ID_CURR = df.SK_ID_CURR
        AND ccb.SK_ID_PREV = df.SK_ID_PREV
        AND ccb.LOAN_TERM = df.LOAN_TERM;
    
"""
con.execute(query)

#### **Gross Default Balance Column**

In [44]:
query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN GROSS_DEFAULT_BALANCE;
"""
con.execute(query)

query = """
    ALTER TABLE credit_card_balance
    ADD COLUMN GROSS_DEFAULT_BALANCE FLOAT DEFAULT 0;
"""
con.execute(query)

query = f"""
    WITH def_bal AS (
        SELECT 
            SK_ID_CURR, SK_ID_PREV,  LOAN_TERM, 
            FIRST_VALUE(AMT_BALANCE) OVER (
                PARTITION BY SK_ID_CURR, SK_ID_PREV, DEFAULT_COUNTER
                ORDER BY LOAN_TERM
            ) AS DEF_BALANCE
        FROM credit_card_balance
        WHERE DEFAULT_DATE IS NOT NULL
    )
    
    UPDATE credit_card_balance AS ccb
    SET GROSS_DEFAULT_BALANCE = df.DEF_BALANCE
    FROM def_bal df
    WHERE ccb.SK_ID_CURR = df.SK_ID_CURR
        AND ccb.SK_ID_PREV = df.SK_ID_PREV
        AND ccb.LOAN_TERM = df.LOAN_TERM;
    
"""
con.execute(query)

#### **Delinquency Indicator**

In [45]:
query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN DELINQUENCY_INDICATOR
"""
con.execute(query)

query = """
    ALTER TABLE credit_card_balance
    ADD COLUMN DELINQUENCY_INDICATOR BOOLEAN DEFAULT FALSE;
"""
con.execute(query)

query = f"""
    WITH delinquency_flags AS (
    SELECT 
        SK_ID_CURR, SK_ID_PREV, LOAN_TERM,
        SUM(CASE WHEN MONTHS_IN_ARREARS >= {DELINQUENCY_THRESHOLD} THEN 1 ELSE 0 END) 
        OVER (
            PARTITION BY SK_ID_CURR, SK_ID_PREV
            ORDER BY LOAN_TERM
            ROWS BETWEEN {STICKINESS_PERIOD - 1} PRECEDING AND CURRENT ROW
        ) AS ARREARS_ROLLING
    FROM credit_card_balance
    )

    UPDATE credit_card_balance ccb
    SET DELINQUENCY_INDICATOR = (
        CASE 
            WHEN df.ARREARS_ROLLING >= {STICKINESS_PERIOD} THEN TRUE 
            ELSE FALSE 
        END
    )
    FROM delinquency_flags df
    WHERE ccb.SK_ID_CURR = df.SK_ID_CURR
    AND ccb.SK_ID_PREV = df.SK_ID_PREV
    AND ccb.LOAN_TERM = df.LOAN_TERM;
"""
con.execute(query)

#### **Inactivity Indicator**

In [46]:
query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN INACTIVE_INDICATOR
"""
con.execute(query)

query = """
    ALTER TABLE credit_card_balance
    ADD COLUMN INACTIVE_INDICATOR BOOLEAN DEFAULT FALSE;
"""
con.execute(query)

query = f"""
    WITH inactive_flags AS (
    SELECT 
        SK_ID_CURR, SK_ID_PREV, LOAN_TERM,
        SUM(CASE WHEN (AMT_BALANCE = 0 AND AMT_DRAWINGS_CURRENT = 0) THEN 1 ELSE 0 END) 
        OVER (
            PARTITION BY SK_ID_CURR, SK_ID_PREV
            ORDER BY LOAN_TERM
            ROWS BETWEEN {INACTIVITY_PERIOD - 1} PRECEDING AND CURRENT ROW
        ) AS INACTIVITY_ROLLING
    FROM credit_card_balance
    )

    UPDATE credit_card_balance ccb
    SET INACTIVE_INDICATOR = (
        CASE 
            WHEN df.INACTIVITY_ROLLING >= {INACTIVITY_PERIOD} THEN TRUE 
            ELSE FALSE 
        END
    )
    FROM inactive_flags df
    WHERE ccb.SK_ID_CURR = df.SK_ID_CURR
    AND ccb.SK_ID_PREV = df.SK_ID_PREV
    AND ccb.LOAN_TERM = df.LOAN_TERM;
"""
con.execute(query)

#### **Transactor Indicator**

In [47]:
query = """
    ALTER TABLE credit_card_balance
    DROP COLUMN TRANSACTOR_INDICATOR
"""
con.execute(query)

query = """
    ALTER TABLE credit_card_balance
    ADD COLUMN TRANSACTOR_INDICATOR BOOLEAN DEFAULT FALSE;
"""
con.execute(query)

query = f"""
    WITH transactor_flags AS (
    SELECT 
        SK_ID_CURR, SK_ID_PREV, LOAN_TERM,
        SUM(
            CASE 
                WHEN (AMT_BALANCE = 0 
                      AND AMT_DRAWINGS_CURRENT > 0 
                      AND INACTIVE_INDICATOR = FALSE)
                THEN 1 ELSE 0 
            END
        ) OVER (
            PARTITION BY SK_ID_CURR, SK_ID_PREV
            ORDER BY LOAN_TERM
            ROWS BETWEEN {TRANSACTOR_PERIOD - 1} PRECEDING AND CURRENT ROW
        ) AS TRANSACTOR_ROLLING
    FROM credit_card_balance
    )
    UPDATE credit_card_balance
    SET TRANSACTOR_INDICATOR = TRUE
    WHERE (SK_ID_CURR, SK_ID_PREV, LOAN_TERM) IN (
        SELECT SK_ID_CURR, SK_ID_PREV, LOAN_TERM
        FROM transactor_flags
        WHERE TRANSACTOR_ROLLING >= {TRANSACTOR_PERIOD}
    );
"""
con.execute(query)

# **Eligibility Filtering**

To align with regulatory modelling scope, a filtering layer is applied to exclude non-eligible records:
- Closed accounts with zero balance are excluded (not considered credit exposures);
- Fraudulent accounts and synthetic records (e.g., dummy or staff test accounts) are removed;
- Accounts with incomplete lifecycle information (e.g., unknown default date or missing limit) are excluded from MRD.

#### **Missing Credit Limits**

In [48]:
query = f"""
        SELECT COUNT(DISTINCT SK_ID_PREV) AS total_records
        FROM credit_card_balance
        WHERE AMT_CREDIT_LIMIT_ACTUAL <= 0;
    """

con.sql(query)

┌───────────────┐
│ total_records │
│     int64     │
├───────────────┤
│             0 │
└───────────────┘

In [49]:
query = """
    DELETE FROM credit_card_balance
    WHERE SK_ID_PREV IN (
        SELECT DISTINCT(SK_ID_PREV)
        FROM credit_card_balance
        WHERE AMT_CREDIT_LIMIT_ACTUAL <= 0 
    );
"""

con.execute(query)

In [50]:
query = f"""
        SELECT COUNT(DISTINCT SK_ID_PREV) AS total_records
        FROM credit_card_balance
        WHERE AMT_CREDIT_LIMIT_ACTUAL <= 0;
    """

con.sql(query)

┌───────────────┐
│ total_records │
│     int64     │
├───────────────┤
│             0 │
└───────────────┘

# **Target Variable Engineering**

For each account at each snapshot date (reporting date), a rolling 12-month observation window is generated to capture model targets:
- Default Flag: Whether a default occurred within the next 12 months;
- Default Type: Further breakdown (e.g., DPD-driven vs UTP);
- Vintage Date: The origination period of the account (used in cohort-based PD modelling);
- Gross Default Balance: Exposure outstanding at point of default (used in LGD/EAD calibration).

#### **Origination Date Column**

In [51]:
query = f"""
    ALTER TABLE credit_card_balance
    DROP COLUMN ORIGINATION_DATE;
"""
con.execute(query)

query = f"""
    ALTER TABLE credit_card_balance
    ADD COLUMN ORIGINATION_DATE DATE DEFAULT NULL;
"""
con.execute(query)

query = f"""
    WITH orig_date AS (
    SELECT
        SK_ID_CURR,SK_ID_PREV, LOAN_TERM,
        FIRST_VALUE(CALENDAR_DATE) OVER (
            PARTITION BY SK_ID_CURR, SK_ID_PREV
            ORDER BY LOAN_TERM
        ) AS ORIGINATION_DATE
    FROM credit_card_balance
    )
    
    UPDATE credit_card_balance AS ccb
    SET ORIGINATION_DATE = (
        SELECT d.ORIGINATION_DATE
        FROM orig_date d
        WHERE ccb.SK_ID_CURR = d.SK_ID_CURR
        AND ccb.SK_ID_PREV = d.SK_ID_PREV
        AND ccb.LOAN_TERM = d.LOAN_TERM
    )

"""
con.execute(query)

#### **12M Outcome Columns**

In [52]:
col_dict = {
    "12M_DELINQUENCY_INDICATOR": ["BOOLEAN", "DELINQUENCY_INDICATOR"],
    "12M_DEFAULT_OUTCOME": ["BOOLEAN", "DEFAULT_INDICATOR"],
    "12M_DEFAULT_COUNTER": ["INTEGER", "DEFAULT_COUNTER"],
    "12M_DEFAULT_TYPE": ["VARCHAR(10)", "DEFAULT_TYPE"],
    "12M_DEFAULT_DATE": ["DATE", "DEFAULT_DATE"],
    "12M_BALANCE": ["FLOAT", "AMT_BALANCE"],
    "12M_GROSS_DEFAULT_BALANCE": ["FLOAT", "GROSS_DEFAULT_BALANCE"],
    "12M_WO_OUTCOME": ["BOOLEAN", "WO_INDICATOR"],
}

for col, col_values in col_dict.items():
    query = f"""
        ALTER TABLE credit_card_balance
        DROP COLUMN "{col}";
    """
    con.execute(query)

    query = f"""
        ALTER TABLE credit_card_balance
        ADD COLUMN "{col}" {col_values[0]} DEFAULT NULL;
    """
    con.execute(query)

    query = f"""
        WITH def_outcome AS (
        SELECT
            SK_ID_CURR, SK_ID_PREV, LOAN_TERM,
            LAG({col_values[1]}, 12, NULL) OVER (
                PARTITION BY SK_ID_CURR, SK_ID_PREV
                ORDER BY LOAN_TERM DESC
            ) AS "{col}",
        FROM credit_card_balance
        )
        
        UPDATE credit_card_balance AS ccb
            SET "{col}" = d."{col}"
            FROM def_outcome d
            WHERE ccb.SK_ID_CURR = d.SK_ID_CURR
        AND ccb.SK_ID_PREV = d.SK_ID_PREV
        AND ccb.LOAN_TERM = d.LOAN_TERM;
    """
    con.execute(query)

### **Segmentation and Enrichment**

To enable risk differentiation and portfolio segmentation:
- Credit bureau attributes are joined to the MRD (e.g., external score bands, income bands, adverse flags);
- Internal attributes are included (e.g., segment, product hierarchy, customer type).

These segmentation variables are critical in ensuring both model discriminatory power and appropriate benchmarking across segments and risk bands, in line with IRB requirements.

# **Model-Ready Data (MRD)**

As part of the credit risk modelling process, data engineering plays a critical role in ensuring inputs are accurate, interpretable, and fit-for-purpose. This section outlines how raw and integrated data are systematically transformed to support development of regulatory models for both IFRS 9 and Advanced IRB (A-IRB) frameworks.

The MRD is derived from the CCD set through a series of structured data preparation steps aimed at engineering variables, flagging modelling events, and applying business rules. This process ensures each record represents a valid and usable snapshot for statistical modelling purposes.

## **Example**

In [53]:
query = f"""
        SELECT *
        FROM credit_card_balance
        -- WHERE SK_ID_CURR = 202468
        WHERE SK_ID_CURR = 387909
        ORDER BY SK_ID_PREV, SK_ID_CURR, LOAN_TERM
    """

test_df = con.sql(query).df()
test_df.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,TRANSACTOR_INDICATOR,ORIGINATION_DATE,12M_DELINQUENCY_INDICATOR,12M_DEFAULT_OUTCOME,12M_DEFAULT_COUNTER,12M_DEFAULT_TYPE,12M_DEFAULT_DATE,12M_BALANCE,12M_GROSS_DEFAULT_BALANCE,12M_WO_OUTCOME
0,1000123,387909,-9,0.0,45000,NaN,0.000,NaN,NaN,0.0,...,False,2012-04-01,<NA>,<NA>,<NA>,None,NaT,NaN,NaN,<NA>
1,1000123,387909,-8,0.0,180000,0.0,43630.470,0.0,43630.470,0.0,...,False,2012-04-01,<NA>,<NA>,<NA>,None,NaT,NaN,NaN,<NA>
2,1000123,387909,-7,0.0,180000,0.0,875.655,0.0,0.000,2250.0,...,True,2012-04-01,<NA>,<NA>,<NA>,None,NaT,NaN,NaN,<NA>
3,1000123,387909,-6,0.0,180000,0.0,178234.155,0.0,178234.155,0.0,...,True,2012-04-01,<NA>,<NA>,<NA>,None,NaT,NaN,NaN,<NA>
4,1000123,387909,-5,0.0,180000,0.0,0.000,0.0,0.000,7915.5,...,False,2012-04-01,<NA>,<NA>,<NA>,None,NaT,NaN,NaN,<NA>


In [54]:
query = """
    SELECT column_name, data_type, ordinal_position, column_default, is_nullable,  character_maximum_length
    FROM information_schema.columns
    WHERE table_name = 'credit_card_balance';
"""

con.sql(query)

┌────────────────────────────┬───────────┬──────────────────┬──────────────────────┬─────────────┬──────────────────────────┐
│        column_name         │ data_type │ ordinal_position │    column_default    │ is_nullable │ character_maximum_length │
│          varchar           │  varchar  │      int32       │       varchar        │   varchar   │          int32           │
├────────────────────────────┼───────────┼──────────────────┼──────────────────────┼─────────────┼──────────────────────────┤
│ SK_ID_PREV                 │ BIGINT    │                1 │ NULL                 │ YES         │                     NULL │
│ SK_ID_CURR                 │ BIGINT    │                2 │ NULL                 │ YES         │                     NULL │
│ MONTHS_BALANCE             │ BIGINT    │                3 │ NULL                 │ YES         │                     NULL │
│ AMT_BALANCE                │ DOUBLE    │                4 │ NULL                 │ YES         │                    

In [55]:
con.close()